<a href="https://colab.research.google.com/github/defyingYang/Bert---Sentiment-Analysis/blob/main/HW2_BERT_112403539.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 情緒分析

資料集: [Learning Word Vectors for Sentiment Analysis](https://aclanthology.org/P11-1015.pdf)

程式碼參考自: [huggingface](https://huggingface.co/)

> **資料集說明**

Large Movie Review Dataset. This is a dataset for binary sentiment classification containing substantially more data than previous benchmark datasets. We provide a set of 25,000 highly polar movie reviews for training, and 25,000 for testing. There is additional unlabeled data for use as well.

>**本次介紹模型為BERT**

[BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://github.com/google-research/bert)

程式碼參考自: [huggingface](https://huggingface.co/)

![](https://i.imgur.com/spiKPbQ.png)


**訓練一個BERT分類模型，輸入是一句話，辨識出這句話的情緒傾向。**




這個colab的程式來完成訓練BERT分類模型。**要跑得出來Testing Accuracy, 須完成TODO1-7**

本次作業著重在學習pytorch的使用方法及Transformer家族中BERT的應用。

> **作業限制**

1. 不要動資料集、不要在訓練時偷看 test data
2. 模型可自行多嘗試不同組合，**不能只用已經pretrained好的模型**，一定再用新的資料訓練過
3. **不要抄襲**


### 資料集下載

- 資料集說明 :
  - text: a string feature.
  - label: a classification label, with possible values including neg (0), pos (1).

## 安裝與載入所需套件

In [ ]:
!pip install datasets transformers

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from transformers.models.bert.modeling_bert import BertPreTrainedModel, BertModel
from sklearn.model_selection import train_test_split
import torch
import torch.nn.functional as Fun
import transformers
import matplotlib.pyplot as plt
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore') # setting ignore as a parameter
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

## 一些模型會用到的小函數

* TODO1: 完成get_pred()
> 從logits的dimension=1去取得結果中數值最高者當做預測結果
* TODO2: 完成cal_metrics()
> 透過將tensor轉為numpy，可使用sklearn的套件算出acc, f1_score, recall及precision

In [ ]:
# get predict result
def get_pred(logits):
  predict = torch.argmax(logits, dim=1)
  return predict


# calculate confusion metrics
def cal_metrics(pred, ans):
  predict = pred.detach().cpu().numpy()
  answer = ans.detach().cpu().numpy()
  acc = accuracy_score(answer, predict,)
  f1 = f1_score(answer, predict, average = 'macro')
  rec = recall_score(answer, predict, average = 'macro')
  prec = precision_score(answer, predict, average = 'macro')
  return acc,f1,rec,prec

In [ ]:
# save model to path
def save_checkpoint(save_path, model):
  if save_path == None:
      return
  torch.save(model.state_dict(), save_path)
  print(f'Model saved to ==> {save_path}')

# load model from path
def load_checkpoint(load_path, model, device):
  if load_path==None:
      return
  state_dict = torch.load(load_path, map_location=device)
  print(f'Model loaded from <== {load_path}')

  model.load_state_dict(state_dict)
  return model

## 載入資料

這個資料集有分成train, test, unsupervised。

這次我們只使用train及test資料，且我們需要將原資料重新進行分割。

將兩份資料合併後切割成 3:1:1 或是 8:1:1 的 train/val/test 資料集。

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

看一下資料格式長怎樣

In [ ]:
dataset

In [ ]:
dataset['train'][0]

* TODO3:把資料拿出來後，將train及test合併，重新切割後，儲存下來。

In [ ]:
import pandas as pd

all_df = [] # a list to save all data

train_data = pd.DataFrame(dataset['train'])
test_data = pd.DataFrame(dataset['test'])

all_df = pd.concat([train_data, test_data])

all_df.head()


可以看一下兩個類別分布的比例

In [ ]:
all_df.label.value_counts() / len(all_df)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_data = train_test_split(all_df, random_state=1111, train_size=0.8)
dev_df, test_df = train_test_split(temp_data, random_state=1111, train_size=0.5)
print('# of train_df:', len(train_df))
print('# of dev_df:', len(dev_df))
print('# of test_df data:', len(test_df))

# save data
train_df.to_csv('./train.tsv', sep='\t', index=False)
dev_df.to_csv('./val.tsv', sep='\t', index=False)
test_df.to_csv('./test.tsv', sep='\t', index=False)

### 自定義 Dataset，將tokenzie的步驟放進去

In [ ]:
from transformers import AutoTokenizer

# 1. 確保你真的「造出」了這台機器（加上括號並賦值）
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 2. 測試文字
test_text = "I love this movie!"

# 3. 使用現代化的「直接呼叫」方式，這比 encode_plus 更不容易報錯
inputs = tokenizer(
    test_text,
    padding='max_length',
    max_length=10,
    truncation=True,
    return_tensors=None # 這裡先拿 list，方便看結果
)

# 4. 印出結果
print("Keys:", inputs.keys())
print("IDs:", inputs['input_ids'])
print("Mask:", inputs['attention_mask'])
print("Type:", inputs['token_type_ids'])

* TODO4: 完成tokenize()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch
import torch.nn.functional as Fun

# Using Dataset to build DataLoader
class CustomDataset(Dataset):
  def __init__(self, mode, df, specify, args):
    assert mode in ["train", "val", "test"]  # 一般會切三份
    self.mode = mode
    self.df = df
    self.specify = specify # specify column of data (the column U use for predict)
    if self.mode != 'test':
      self.label = df['label']
    self.tokenizer = AutoTokenizer.from_pretrained(args["config"])
    self.max_len = args["max_len"]
    self.num_class = args["num_class"]

  def __len__(self):
    return len(self.df)

  # transform label to one_hot label (if num_class > 2)
  def one_hot_label(self, label):
    return Fun.one_hot(torch.tensor(label), num_classes = self.num_class)

  # transform text to its number
  def tokenize(self,input_text):
    inputs = self.tokenizer(
        input_text,
        padding='max_length',
        max_length=self.max_len,
        truncation=True,
        return_tensors=None
    )

    ids = inputs['input_ids']
    mask = inputs['attention_mask']
    token_type_ids =inputs['token_type_ids']

    return ids, mask, token_type_ids

  # get single data
  def __getitem__(self, index):

    sentence = str(self.df[self.specify][index])
    ids, mask, token_type_ids = self.tokenize(sentence)


    if self.mode == "test":
        return torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.long), \
            torch.tensor(token_type_ids, dtype=torch.long)
    else:
        if self.num_class > 2:
          return torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.long), \
            torch.tensor(token_type_ids, dtype=torch.long), self.one_hot_label(self.label[index])
        else:
          return torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.long), \
            torch.tensor(token_type_ids, dtype=torch.long), torch.tensor(self.label[index], dtype=torch.long)

## 建立模型

*   自己重新寫分類模型
*   模型訓練與評估的程式碼大致上相同，差別在於模型是否繼續進行梯度下降，以及模型參數是否繼續訓練



* TODO5: 完成BertClassifier
> 在初始化的地方加上dropout, linear layer（等於一層NN），其維度為類別數量；
> 在forward function中把輸入值放進對應層數（bert -> dropout -> classifier）；
> 請注意我們只取用bert輸出的sentence representation去做分類

In [ ]:
import torch.nn as nn
from transformers import BertModel

# BERT Model
class BertClassifier(nn.Module):
  def __init__(self, args):
    super(BertClassifier, self).__init__()
    self.bert = BertModel.from_pretrained(args['config'])
    ##########
    # todo #
    ##########
    self.dropout = torch.nn.Dropout(args.get("dropout", 0.1))
    self.classifier = torch.nn.Linear(self.bert.config.hidden_size, args['num_class'])


  # forward function, data in model will do this
  def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None,
              head_mask=None, inputs_embeds=None, labels=None, output_attentions=None,
              output_hidden_states=None, return_dict=None):
    ##########
    # todo #
    ##########
    outputs = self.bert(
        input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
        return_dict=True
    )

    pooled_output = outputs.pooler_output
    pooled_output = self.dropout(pooled_output)
    logits = self.classifier(pooled_output)

    return logits



這是已經寫好的evaluate()，train()會跟他很像

(# 在評估模型的時候，不需要梯度下降)

In [ ]:
# evaluate dataloader
def evaluate(model, data_loader, device):
  val_loss, val_acc, val_f1, val_rec, val_prec = 0.0, 0.0, 0.0, 0.0, 0.0
  step_count = 0
  loss_fct = torch.nn.CrossEntropyLoss()
  model.eval()
  with torch.no_grad():
    for data in data_loader:
      ids, masks, token_type_ids, labels = [t.to(device) for t in data]

      logits = model(input_ids = ids,
              token_type_ids = token_type_ids,
              attention_mask = masks)
      acc, f1, rec, prec = cal_metrics(get_pred(logits), labels)
      loss = loss_fct(logits, labels)

      val_loss += loss.item()
      val_acc += acc
      val_f1 += f1
      val_rec += rec
      val_prec += prec
      step_count+=1

    val_loss = val_loss / step_count
    val_acc = val_acc / step_count
    val_f1 = val_f1 / step_count
    val_rec = val_rec / step_count
    val_prec = val_prec / step_count

  return val_loss, val_acc, val_f1, val_rec, val_prec

## 開始訓練

### 定義你的 Hyperparameters

* 如果電腦的記憶體不夠可以試著減少 batch_size
* 因為我們採用現有的模型去fine-tune，所以一般不需要設太多 epochs
* config 就是我們所使用的現有模型，可以自己找適合的做替換
* 這份 work 是做二分類，所以 num_class 為 2
* 如果你的模型 overfit 了，可以把 dropout 調高
* 可以試著調高或調低 learning_rate，這會影響他的學習速度（跨步的大小）
* 你應該先檢閱你的資料再來決定 max_len （但 BERT 最大只吃到 512）

In [ ]:
from datetime import datetime
parameters = {
    "num_class": 2,
    "time": str(datetime.now()).replace(" ", "_"),
    # Hyperparameters
    "model_name": 'BERT',
    "config": 'bert-base-uncased',
    "learning_rate": 1e-5,
    "epochs": 3,
    "max_len": 512,
    "batch_size": 36,
    "dropout": 0.1,
}

### 載入資料

讀入資料並傳入自訂的Dataset以自訂資料格式

之後傳入DataLoader以利後續訓練進行（將資料批次化以免記憶體爆掉）

(# 你可以決定要sample部分資料還是全部都丟進去)

In [ ]:
import transformers
import pandas as pd

# load training data
train_df = pd.read_csv('./train.tsv', sep = '\t').sample(4000).reset_index(drop=True)
train_dataset = CustomDataset('train', train_df, 'text', parameters)
train_loader = DataLoader(train_dataset, batch_size=parameters['batch_size'], shuffle=True)

# load validation data
val_df = pd.read_csv('./val.tsv', sep = '\t').sample(500).reset_index(drop=True)
val_dataset = CustomDataset('val', val_df, 'text', parameters)
val_loader = DataLoader(val_dataset, batch_size=parameters['batch_size'], shuffle=True)

### 初始化模型
*   載入模型（這邊會使用已經訓練過的模型，Fine-tune我們的資料集）
*   定義Optimization
  *   通常用Adam就可以了，你也可以換SGD之類的試看看
  *   可以自己看需不需要加scheduler（可以自己寫一個function，也可以直接套用現有的function）
  
  ［請記得pytorch中是以step去計算，想要用epoch去訂定需自行換算］




In [ ]:
from transformers import get_cosine_schedule_with_warmup

transformers.logging.set_verbosity_error() # close the warning message

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertClassifier(args=parameters).to(device)
loss_fct = torch.nn.CrossEntropyLoss() # we use cross entrophy loss

## You can custom your optimizer (e.g. SGD .etc) ##
# 這邊嘗試用AdamW
optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['learning_rate'], weight_decay=0.01)

## You also can add your custom scheduler ##
num_train_steps = len(train_loader) * parameters["epochs"]
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * num_train_steps), num_training_steps=num_train_steps, num_cycles=1)

* 因為是做分類任務，所以這裡用CrossEntrophyLoss
* 由於在pytorch中，CrossEntrophyLoss吃的input是未經softmax的值，所以在模型中不必加入softmax
* 但若在後期想取得各類別實際機率就要經過softmax轉換
* logits的數值不等於機率 ！！但一般而言，logits中較大者會與經過softmax轉換後的結果一致

* TODO6: 完成訓練，可參照evaluate()並稍作調整以完成訓練

In [ ]:
# Start training
import time
metrics = ['loss', 'acc', 'f1', 'rec', 'prec']
mode = ['train_', 'val_']
record = {s+m :[] for s in mode for m in metrics}

for epoch in range(parameters["epochs"]):

    st_time = time.time()
    train_loss, train_acc, train_f1, train_rec, train_prec = 0.0, 0.0, 0.0, 0.0, 0.0
    step_count = 0

    ##########
    # todo #
    ##########
    for data in train_loader:
      ids, masks, token_type_ids, labels = [t.to(device) for t in data]

      optimizer.zero_grad()

      logits = model(input_ids = ids,
                token_type_ids = token_type_ids,
                attention_mask = masks)

      loss = loss_fct(logits, labels)
      loss.backward()
      optimizer.step()
      scheduler.step()

      train_loss += loss.item()
      acc, f1, rec, prec = cal_metrics(get_pred(logits), labels)
      train_acc += acc
      train_f1 += f1
      train_rec += rec
      train_prec += prec
      step_count+=1





    # evaluate the model performace on val data after finishing an epoch training
    val_loss, val_acc, val_f1, val_rec, val_prec = evaluate(model, val_loader, device)

    train_loss = train_loss / step_count
    train_acc = train_acc / step_count
    train_f1 = train_f1 / step_count
    train_rec = train_rec / step_count
    train_prec = train_prec / step_count

    print('[epoch %d] cost time: %.4f s'%(epoch + 1, time.time() - st_time))
    print('         loss     acc     f1      rec    prec')
    print('train | %.4f, %.4f, %.4f, %.4f, %.4f'%(train_loss, train_acc, train_f1, train_rec, train_prec))
    print('val  | %.4f, %.4f, %.4f, %.4f, %.4f\n'%(val_loss, val_acc, val_f1, val_rec, val_prec))

    # record training metrics of each training epoch
    record['train_loss'].append(train_loss)
    record['train_acc'].append(train_acc)
    record['train_f1'].append(train_f1)
    record['train_rec'].append(train_rec)
    record['train_prec'].append(train_prec)

    record['val_loss'].append(val_loss)
    record['val_acc'].append(val_acc)
    record['val_f1'].append(val_f1)
    record['val_rec'].append(val_rec)
    record['val_prec'].append(val_prec)

In [ ]:
# save model
save_checkpoint('./bert.pt' , model)

### 畫圖

In [ ]:
# draw learning curve
import matplotlib.pyplot as plt
def draw_pics(record, name, img_save=False, show=False):
    EPOCHS = parameters["epochs"]
    x_ticks = range(1, EPOCHS+1)

    plt.figure(figsize=(6, 3))

    plt.plot(x_ticks, record['train_'+name], '-o', color='lightskyblue',
             markeredgecolor="teal", markersize=3, markeredgewidth=1, label = 'Train')
    plt.plot(x_ticks, record['val_'+name], '-o', color='pink',
             markeredgecolor="salmon", markersize=3, markeredgewidth=1, label = 'Val')
    plt.grid(color='lightgray', linestyle='--', linewidth=1)

    plt.title('Model', fontsize=14)
    plt.ylabel(name, fontsize=12)
    plt.xlabel('Epoch', fontsize=12)
    plt.xticks(x_ticks, fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(loc='lower right' if not name.lower().endswith('loss') else 'upper right')

    if img_save:
        plt.savefig(name+'.png', transparent=False, dpi=300)
    if show:
        plt.show()

    plt.close()

In [ ]:
draw_pics(record, 'loss', img_save=False, show=True)

In [ ]:
draw_pics(record, 'acc', img_save=False, show=True)

In [ ]:
draw_pics(record, 'f1', img_save=False, show=True)

In [ ]:
draw_pics(record, 'rec', img_save=False, show=True)

In [ ]:
draw_pics(record, 'prec', img_save=False, show=True)

## 預測結果

預測單筆（跟評估的程式大同小異）




In [ ]:
def Softmax(x):
  return torch.exp(x) / torch.exp(x).sum()
# label to class
def label2class(label):
  l2c = {0:'negative', 1:'positive'}
  return l2c[label]

* TODO7: 完成predict_one()

In [ ]:
# predict single sentence, return each-class's probability and predicted class
def predict_one(query, model):

  inputs = tokenizer(
    query,
    padding='max_length',
    max_length=10,
    truncation=True,
    return_tensors=None # 這裡先拿 list，方便看結果
  ).to(device)

  ids = inputs['input_ids']
  masks = inputs['attention_mask']
  token_type_ids =inputs['token_type_ids']

  logits = model(
      input_ids = ids,
      token_type_ids = token_type_ids,
      attention_mask = masks)

  probs = Softmax(logits)

  pred = torch.argmax(probs[0], dim=0).item()

  return probs, pred

In [ ]:
# you can load model from existing result
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
init_model = BertClassifier(args=parameters).to(device) # build an initial model
model = load_checkpoint('./bert.pt', init_model, device).to(device) # and load the weight of model from specify file

In [ ]:
%%time
probs, pred = predict_one("This movie doesn't attract me", model)
print(label2class(pred))

:你也可以像evaluate function一樣，把它寫成dataloader的形式

In [ ]:
# predict dataloader
def predict(data_loader, model):

  tokenizer = AutoTokenizer.from_pretrained(parameters['config'])
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  total_probs, total_pred = [], []
  model.eval()
  with torch.no_grad():
    for data in data_loader:
      input_ids, attention_mask, \
      token_type_ids = [t.to(device) for t in data]

      # forward pass
      logits = model(input_ids, attention_mask, token_type_ids)
      probs = Softmax(logits) # get each class-probs
      label_index = torch.argmax(probs[0], dim=0)
      pred = label_index.item()

      total_probs.append(probs)
      total_pred.append(pred)

  return total_probs, total_pred

In [ ]:
# load testing data
test_df = pd.read_csv('./test.tsv', sep = '\t').sample(500).reset_index(drop=True)
test_dataset = CustomDataset('test', test_df, 'text', parameters)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

total_probs, total_pred = predict(test_loader, model)
res = test_df.copy()
# add predict class of origin file
res['pred'] = total_pred

# save result
res.to_csv('./result.tsv', sep='\t', index=False)

In [ ]:
res.head(5)

In [ ]:
correct = 0
for idx, pred in enumerate(res['pred']):
  if pred == res['label'][idx]:
    correct += 1
print('test accuracy = %.4f'%(correct/len(test_df)))